# Rally Oil Price in Plotly

Replica del gráfico `rally-oil-price` usando Plotly y la misma fuente FRED (`DCOILBRENTEU`).

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path

FRED_BRENT_URL = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=DCOILBRENTEU"

In [ ]:
df = pd.read_csv(FRED_BRENT_URL)
df = df.rename(columns={"DATE": "date", "DCOILBRENTEU": "price", "observation_date": "date"})
df["date"] = pd.to_datetime(df["date"])
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df = df.dropna(subset=["date", "price"])
df = df[df["price"] > 0].sort_values("date").reset_index(drop=True)
df["year"] = df["date"].dt.year
df.head()

In [ ]:
series = []

for year, group in df.groupby("year", sort=True):
    group = group.copy().reset_index(drop=True)
    base_price = group.loc[0, "price"]
    group["day"] = range(1, len(group) + 1)
    group["changePct"] = ((group["price"] / base_price) - 1) * 100
    series.append({
        "year": int(year),
        "firstDate": group.loc[0, "date"].strftime("%Y-%m-%d"),
        "lastDate": group.loc[len(group) - 1, "date"].strftime("%Y-%m-%d"),
        "basePrice": round(float(base_price), 2),
        "lastPrice": round(float(group.loc[len(group) - 1, "price"]), 2),
        "points": group[["day", "date", "price", "changePct"]].copy()
    })

current_year = series[-1]["year"]
current_year

In [ ]:
fig = go.Figure()

for entry in series:
    year = entry["year"]
    points = entry["points"]

    if year == current_year:
        color = "#ffd166"
        width = 3.25
    elif year in (1999, 2020):
        color = "rgba(255,255,255,0.22)"
        width = 1.1
    else:
        color = "rgba(255,255,255,0.22)"
        width = 1.1

    fig.add_trace(
        go.Scatter(
            x=points["day"],
            y=points["changePct"],
            mode="lines",
            name=str(year),
            line=dict(color=color, width=width),
            hovertemplate=(
                f"<b>{year}</b><br>" +
                "Trading day %{x}<br>" +
                "Change %{y:.2f}%<br>" +
                "<extra></extra>"
            ),
            showlegend=False,
        )
    )

fig.update_layout(
    template="plotly_dark",
    title="Oil 2026: the most aggressive rally in 30 years",
    paper_bgcolor="#050505",
    plot_bgcolor="#050505",
    font=dict(family="Arial", color="#f2f2f2"),
    margin=dict(l=62, r=34, t=80, b=60),
    xaxis=dict(
        title="Trading day",
        tickmode="array",
        tickvals=[50, 100, 150, 200, 250],
        gridcolor="rgba(255,255,255,0.00)",
        zeroline=False,
    ),
    yaxis=dict(
        title=None,
        gridcolor="rgba(255,255,255,0.09)",
        zeroline=False,
    ),
)

fig.show()

In [ ]:
# Opcional: guardar HTML interactivo al lado del notebook
out = Path("rally_oil_price_plotly.html")
fig.write_html(out, include_plotlyjs="cdn")
out